In [ ]:
!pip install matplotlib-venn
!pip install datasets transformers evaluate
!pip install sentence-transformers
!pip install fsspec==2023.6.0
!pip install rouge_score
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    get_scheduler
)
from torch.optim import AdamW
from datasets import load_dataset
import evaluate
from tqdm import tqdm
import numpy as np
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


def prepare_dataset():
    dataset = load_dataset("cnn_dailymail", "3.0.0")

    def preprocess(examples):
        inputs = ["summarize: " + doc for doc in examples["article"]]
        targets = examples["highlights"]
        return {"inputs": inputs, "targets": targets}

    dataset = dataset.map(preprocess, batched=True, remove_columns=["article", "highlights", "id"])
    return dataset["train"], dataset["validation"], dataset["test"]

train_dataset, val_dataset, test_dataset = prepare_dataset()


model_name = "allenai/primera"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)


def data_collator(batch):
    inputs = [item["inputs"] for item in batch]
    targets = [item["targets"] for item in batch]

    model_inputs = tokenizer(
        inputs,
        max_length=1024,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            max_length=256,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        ).input_ids

    labels[labels == tokenizer.pad_token_id] = -100

    return {
        "input_ids": model_inputs["input_ids"],
        "attention_mask": model_inputs["attention_mask"],
        "labels": labels
    }


def train_model(model, train_dataset, val_dataset, epochs=1, batch_size=2):
    loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=data_collator
    )

    optimizer = AdamW(model.parameters(), lr=5e-5)
    scheduler = get_scheduler(
        "linear",
        optimizer=optimizer,
        num_warmup_steps=50,
        num_training_steps=len(loader) * epochs
    )

    model.train()
    for epoch in range(epochs):
        print(f"\nEpoch {epoch + 1}/{epochs}")
        total_loss = 0
        for batch in tqdm(loader, desc="Training"):
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            total_loss += loss.item()

        print(f"Average Loss: {total_loss / len(loader):.4f}")
    return model


print("\nTraining PRIMERA on partial CNN/DailyMail...")
model = train_model(
    model,
    train_dataset.select(range(1000)),
    val_dataset.select(range(200)),
    epochs=1,
    batch_size=2
)


rouge = evaluate.load("rouge")

def evaluate_model(model, test_dataset, num_samples=50):
    model.eval()
    predictions, references = [], []
    subset = test_dataset.select(range(num_samples))

    with torch.no_grad():
        for example in tqdm(subset, desc="Evaluating"):
            inputs = tokenizer(
                example["inputs"],
                return_tensors="pt",
                max_length=1024,
                padding="max_length",
                truncation=True
            ).to(device)

            summary_ids = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_length=256,
                num_beams=4,
                early_stopping=True
            )
            summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
            predictions.append(summary)
            references.append(example["targets"])

    return rouge.compute(predictions=predictions, references=references, use_stemmer=True)


print("\nEvaluating PRIMERA...")
results = evaluate_model(model, test_dataset, num_samples=50)


results_df = pd.DataFrame({
    "Model": ["PRIMERA"],
    "ROUGE-1": [results["rouge1"]],
    "ROUGE-2": [results["rouge2"]],
    "ROUGE-L": [results["rougeL"]]
})

print("\nEvaluation Results:")
print(results_df)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 101.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Extracting data files:   0%|          | 0/3 [00:00<?, ?it/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

...

Training:   0%|          | 0/500 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3951: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Training:   0%|          | 1/500 [00:02<21:13,  2.55s/it]/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3951: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Training:   0%|          | 2/500 [00:04<16:10,  1.95s/it]/usr/local/lib/python3.11/dist-packag

Average Loss: 1.9530



Evaluating PRIMERA...


Evaluating: 100%|██████████| 50/50 [01:19<00:00,  1.58s/it]



Evaluation Results:
     Model   ROUGE-1   ROUGE-2   ROUGE-L
0  PRIMERA  0.338067  0.146937  0.242924
